In [ ]:
token = ""

In [ ]:
repo_url = f"https://{token}@github.com/LudwikMadej/WB2-collab-friendly"

!git clone {repo_url}


Cloning into 'WB2-collab-friendly'...
remote: Enumerating objects: 71, done.
remote: Total 71 (delta 0), reused 0 (delta 0), pack-reused 71 (from 1)
Receiving objects: 100% (71/71), 87.72 MiB | 13.59 MiB/s, done.
Resolving deltas: 100% (3/3), done.
Updating files: 100% (61/61), done.


In [ ]:
%cd WB2-collab-friendly

/content/WB2-collab-friendly


In [ ]:
from software.raw_data import get_raw_test_data, get_raw_train_data

In [ ]:
data = get_raw_train_data(1, data_folder_path='data/')
data.head()

/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,0,1,2,3,4,5,6,7,8,9,...,1015,1016,1017,1018,1019,1020,1021,1022,1023,glasses
0,0.038330,0.043945,-0.089844,0.071289,-0.089844,0.031982,0.000488,0.058105,0.029785,-0.032715,...,-0.025879,-0.023926,0.062500,-0.074219,0.103516,0.067871,0.030518,-0.094727,0.095215,0
1,0.049072,0.026367,-0.083008,0.040527,-0.082520,0.046143,0.008545,0.061035,0.043701,-0.043701,...,-0.008911,-0.018066,0.026489,-0.072266,0.100586,0.054443,0.084961,-0.070312,0.070312,0
2,0.003540,0.051758,-0.145508,0.096680,-0.078125,0.041748,-0.016602,0.074219,0.011475,-0.014282,...,-0.026123,-0.024902,0.043701,-0.042969,0.051758,0.058594,0.065430,-0.073730,0.110352,0
3,0.027832,0.046387,-0.112793,0.061279,-0.076660,0.027588,-0.017090,0.068359,0.021484,-0.027588,...,-0.034912,-0.030518,0.039307,-0.082031,0.093262,0.056641,0.057129,-0.076660,0.094238,0
4,0.036133,0.030762,-0.093262,0.070312,-0.071777,0.050537,-0.017334,0.061035,0.006348,-0.037598,...,-0.043213,-0.037354,0.046387,-0.081543,0.069336,0.023926,0.065430,-0.031738,0.095215,0


In [ ]:
X, y = data.iloc[:, :-1].to_numpy(), data.iloc[:, -1]

In [ ]:
import numpy as np

class IterativeCAVDebiaser:
    def __init__(self, max_k=16):
        self.max_k = max_k
        self.cavs_ = []

    def _compute_cav(self, X, y):
        # Rzutujemy na float64, aby uniknąć błędów przy małych wartościach
        X_high = X.astype(np.float64)

        mean_pos = X_high[y == 1].mean(axis=0)
        mean_neg = X_high[y == 0].mean(axis=0)

        cav = mean_pos - mean_neg
        norm = np.linalg.norm(cav)

        # W float16 norma rzędu 1e-4 to już granica błędu.
        # Używamy bezpiecznego progu dla stabilności.
        if norm < 1e-7:
            return None

        return cav / norm

    def debias(self, X, cav):
        # Upewniamy się, że operacje wektorowe zachowują precyzję
        X_high = X.astype(np.float64)
        cav_high = cav.astype(np.float64)

        # Wykonujemy ortogonalizację
        res = X_high - np.dot(X_high, cav_high).reshape(-1, 1) * cav_high

        # Możemy wrócić do oryginalnego typu danych użytkownika na końcu
        return res.astype(X.dtype)

    def fit(self, X, y):
        self.cavs_ = []
        X_current = X.copy()

        for i in range(self.max_k):
            cav_norm = self._compute_cav(X_current, y)

            if cav_norm is None:
                # Jeśli wektor jest zbyt mały, przerywamy i nie dodajemy nic więcej
                break

            self.cavs_.append(cav_norm)
            X_current = self.debias(X_current, cav_norm)

        return self

    def transform(self, X, y=None, k=None):
        if k is None:
            k = len(self.cavs_)

        if k > len(self.cavs_):
            raise ValueError(f"Dostępnych CAV: {len(self.cavs_)}, żądano: {k}")

        X_transformed = X.copy()
        for i in range(k):
            X_transformed = self.debias(X_transformed, self.cavs_[i])

        return X_transformed

    def fit_transform(self, X, y, k=None):
        return self.fit(X, y).transform(X, y=y, k=k)

In [ ]:
icd = IterativeCAVDebiaser()

In [ ]:
X = icd.fit_transform(X, y, k=1)

In [ ]:
for arr in (icd.cavs_):
  print(np.any(arr > 0))

True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True


In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

# Po restarcie środowiska (wykona się automatycznie), stwórz środowisko z Pythonem 3.10
!mamba install -y python=3.10 cuml

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:07
🔁 Restarting kernel...

Looking for: ['python=3.10', 'cuml']

[+] 0.0s
conda-forge/linux-64  ⣾  
conda-forge/noarch    ⣾  [+] 0.1s
conda-forge/linux-64   1%
conda-forge/noarch    ⣾  [+] 0.2s
conda-forge/linux-64   8%
conda-forge/noarch    12%[+] 0.3s
conda-forge/linux-64  16%
conda-forge/noarch    27%[+] 0.4s
conda-forge/linux-64  19%
conda-forge/noarch    35%[+] 0.5s
conda-forge/linux-64  24%
conda-forge/noarch    45%[+] 0.6s
conda-forge/linux-64  29%
conda-forge/noarch    54%[+] 0.7s
conda-forge/linux-64  32%
conda-forge/noarch    60%[+] 0.8s
conda-forge/linux-64  37%
conda-forge/noarch    71%[+] 0.9s
conda-forge/linux-64  43%
conda-forge/noarch    82%[+] 1.0s
conda-forge/linux-64  47%
conda-forge/noarch    92%conda-forge/noarch                                


In [ ]:
!mamba install -c rapidsai -c nvidia -c conda-forge cuml python=3.10 cuda-version=12.0

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

# Modele GPU
from cuml.linear_model import LogisticRegression as cuLogReg
from xgboost import XGBClassifier

from cuml.common import logger

# 4 = Error, 5 = Critical, 6+ = Całkowita cisza
logger.set_level(5)

from warnings import filterwarnings
filterwarnings('ignore')

# --- KONFIGURACJA ---
NUM_LAYERS = 24
MAX_DEBIAS_STEPS = 16
RUNS_PER_STEP = 5
OUTPUT_DIR_DEBIASED = "./checkpoint_02/results"
os.makedirs(OUTPUT_DIR_DEBIASED, exist_ok=True)

def get_metrics(y_true, y_pred, y_prob):
    """Oblicza metryki, zabezpieczając się przed brakiem obu klas w predykcji."""
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0.5
    return {
        'acc': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'auc': auc
    }

def pclarc_step(X, y, cav=None, target_val=None):
    """
    Implementacja PClarc dla wektorów.
    Przesuwa dane wzdłuż CAV do poziomu średniej klasy 'czystej' (y=0).
    """
    if cav is None:
        # Obliczanie CAV w najwyższej precyzji float64
        m_pos = X[y == 1].mean(axis=0)
        m_neg = X[y == 0].mean(axis=0)
        cav = m_pos - m_neg
        norm = np.linalg.norm(cav)

        if norm < 1e-15:
            return X, None, None

        cav /= norm
        # Cel: poziom rzutu klasy negatywnej (uznawanej za czystą)
        target_val = np.dot(m_neg, cav)

    # Rzutowanie i przesunięcie (shift)
    current_projections = np.dot(X, cav)
    shift = target_val - current_projections

    # X_new = X + shift * cav
    X_new = X + np.outer(shift, cav)
    return X_new, cav, target_val

# --- GŁÓWNA PĘTLA BADANIA ---
for layer in tqdm(range(NUM_LAYERS), desc="Warstwy"):
    layer_results = []

    # Ładowanie danych - precyzja float64 dla obliczeń macierzowych
    train_df = get_raw_train_data(layer, 'data/')
    test_df = get_raw_test_data(layer, 'data/')

    X_tr_current = train_df.iloc[:, :-1].values.astype(np.float64)
    y_tr = train_df.iloc[:, -1].astype(int).values
    X_te_current = test_df.iloc[:, :-1].values.astype(np.float64)
    y_te = test_df.iloc[:, -1].astype(int).values

    # Skalowanie początkowe
    scaler = StandardScaler()
    X_tr_current = scaler.fit_transform(X_tr_current)
    X_te_current = scaler.transform(X_te_current)

    # Iteracyjny debiasing PClarc
    for k in tqdm(range(MAX_DEBIAS_STEPS + 1), leave=False, desc="Debiasing step"):

        # Trenujemy 10 modeli dla każdego poziomu k (bez bootstrapu danych)
        for run_id in range(RUNS_PER_STEP):

            models = {
                'XGBoost': XGBClassifier(
                    n_estimators=200,
                    max_depth=4,
                    device="cuda",
                    tree_method="hist",
                    random_state=run_id, # Różne ziarno dla każdego z 10 modeli
                    subsample=0.8,       # Dodajemy lekki szum procesowy dla XGB
                    verbosity=0
                )
            }

            for model_name, model in models.items():
                # Modele GPU pracują na float32
                model.fit(X_tr_current.astype(np.float32), y_tr)

                tr_prob = model.predict_proba(X_tr_current.astype(np.float32))[:, 1]
                te_prob = model.predict_proba(X_te_current.astype(np.float32))[:, 1]

                tr_m = get_metrics(y_tr, (tr_prob > 0.5).astype(int), tr_prob)
                te_m = get_metrics(y_te, (te_prob > 0.5).astype(int), te_prob)

                layer_results.append({
                    'layer_id': layer,
                    'k_steps': k,
                    'run_id': run_id,
                    'model': model_name,
                    **{f'train_{key}': v for key, v in tr_m.items()},
                    **{f'test_{key}': v for key, v in te_m.items()}
                })

        # Przygotowanie danych na kolejny krok k+1 metodą PClarc
        if k < MAX_DEBIAS_STEPS:
            # 1. Wyliczamy parametry PClarc na treningu
            X_tr_current, cav, t_val = pclarc_step(X_tr_current, y_tr)

            if cav is None:
                break

            # 2. Aplikujemy te same parametry na zbiór testowy
            X_te_current, _, _ = pclarc_step(X_te_current, y_te, cav=cav, target_val=t_val)

    # Zapis wyników dla warstwy
    pd.DataFrame(layer_results).to_csv(f"{OUTPUT_DIR_DEBIASED}/{layer:02d}_pclarc_results.csv", index=False)

Warstwy:   0%|          | 0/24 [00:00<?, ?it/s]

Debiasing step:   0%|          | 0/17 [00:00<?, ?it/s]

Debiasing step:   0%|          | 0/17 [00:00<?, ?it/s]

Debiasing step:   0%|          | 0/17 [00:00<?, ?it/s]

Debiasing step:   0%|          | 0/17 [00:00<?, ?it/s]

...

Debiasing step:   0%|          | 0/17 [00:00<?, ?it/s]

Debiasing step:   0%|          | 0/17 [00:00<?, ?it/s]

Debiasing step:   0%|          | 0/17 [00:00<?, ?it/s]

Debiasing step:   0%|          | 0/17 [00:00<?, ?it/s]

Debiasing step:   0%|          | 0/17 [00:00<?, ?it/s]

In [ ]:
import shutil

# 1. Definiujemy nazwę pliku wyjściowego
zip_filename = "pclarc_results_full"
directory_to_zip = OUTPUT_DIR_DEBIASED

# 2. Tworzymy archiwum ZIP
# shutil.make_archive automatycznie doda rozszerzenie .zip
shutil.make_archive(zip_filename, 'zip', directory_to_zip)

print(f"✅ Folder {directory_to_zip} został spakowany do pliku: {zip_filename}.zip")

# 3. Kod do automatycznego pobrania (dla Google Colab)
try:
    from google.colab import files
    files.download(f"{zip_filename}.zip")
    print("🚀 Rozpoczęto pobieranie...")
except ImportError:
    print(f"ℹ️ Jeśli nie jesteś w Colabie, plik znajdziesz w: {os.getcwd()}/{zip_filename}.zip")

✅ Folder ./checkpoint_02/results został spakowany do pliku: pclarc_results_full.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🚀 Rozpoczęto pobieranie...


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from cuml.linear_model import LogisticRegression as cuLogReg
from cuml.common import logger
from cuml.svm import SVC

logger.set_level(5) # Cisza w logach GPU

# --- KONFIGURACJA TESTOWA ---
TEST_LAYER = 0  # Sprawdzamy tylko jedną warstwę
DEBIAS_STEPS_TO_SHOW = 5
C_VALUE = 10 # Silniejsza regularyzacja dla stabilności przy N < D

def quick_metrics(y_true, y_prob):
    y_pred = (y_prob > 0.5).astype(int)
    # Sprawdzenie czy model nie przewiduje tylko jednej klasy
    unique_preds = np.unique(y_pred)

    try:
        auc = roc_auc_score(y_true, y_prob)
    except:
        auc = 0.5

    return {
        'acc': accuracy_score(y_true, y_pred),
        'auc': auc,
        'unique_labels': len(unique_preds)
    }


def pclarc_step(X, y, cav=None, target_val=None):
  """
  Implementacja PClarc dla wektorów.
  Przesuwa dane wzdłuż CAV do poziomu średniej klasy 'czystej' (y=0).
  """
  if cav is None:
      # Obliczanie CAV w najwyższej precyzji float64
      m_pos = X[y == 1].mean(axis=0)
      m_neg = X[y == 0].mean(axis=0)
      cav = m_pos - m_neg
      norm = np.linalg.norm(cav)

      if norm < 1e-15:
          return X, None, None

      cav /= norm
      # Cel: poziom rzutu klasy negatywnej (uznawanej za czystą)
      target_val = np.dot(m_neg, cav)

  # Rzutowanie i przesunięcie (shift)
  current_projections = np.dot(X, cav)
  shift = target_val - current_projections

  # X_new = X + shift * cav
  X_new = X + np.outer(shift, cav)
  return X_new, cav, target_val

# Zakładamy, że funkcje get_raw_... i pclarc_step są zdefiniowane jak wcześniej

print(f"\n=== SZYBKI TEST: WARSTWA {TEST_LAYER} ===")
print(f"{'Krok':<5} | {'Zbiór':<7} | {'ACC':<6} | {'AUC':<6} | {'Klasy':<6}")
print("-" * 45)

# 1. Ładowanie i prep
train_df = get_raw_train_data(TEST_LAYER, 'data/')
test_df = get_raw_test_data(TEST_LAYER, 'data/')

X_tr = train_df.iloc[:, :-1].values.astype(np.float64)
y_tr = train_df.iloc[:, -1].astype(int).values
X_te = test_df.iloc[:, :-1].values.astype(np.float64)
y_te = test_df.iloc[:, -1].astype(int).values

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_te = scaler.transform(X_te)

# 2. Pętla debiasingu
for k in range(DEBIAS_STEPS_TO_SHOW + 1):
    # Model
    model = XGBClassifier(
                    n_estimators=100,
                    max_depth=2,
                    device="cuda",
                    tree_method="hist",
                    random_state=1, # Różne ziarno dla każdego z 10 modeli
                    subsample=0.8,       # Dodajemy lekki szum procesowy dla XGB
                    verbosity=0
                )
    model.fit(X_tr.astype(np.float32), y_tr)

    # Predykcje
    tr_p = model.predict_proba(X_tr.astype(np.float32))[:, 1]
    te_p = model.predict_proba(X_te.astype(np.float32))[:, 1]

    # Metryki
    m_tr = quick_metrics(y_tr, tr_p)
    m_te = quick_metrics(y_te, te_p)

    # Print wyników
    print(f"{k:<5} | Train   | {m_tr['acc']:.3f} | {m_tr['auc']:.3f} | {m_tr['unique_labels']}")
    print(f"{'':<5} | Test    | {m_te['acc']:.3f} | {m_te['auc']:.3f} | {m_te['unique_labels']}")
    print("-" * 45)

    # Krok PClarc
    X_tr, cav, t_val = pclarc_step(X_tr, y_tr)
    if cav is None: break
    X_te, _, _ = pclarc_step(X_te, y_te, cav=cav, target_val=t_val)

    # KLUCZOWE: Ponowne skalowanie po przesunięciu wektorów
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)


=== SZYBKI TEST: WARSTWA 0 ===
Krok  | Zbiór   | ACC    | AUC    | Klasy 
---------------------------------------------
0     | Train   | 0.921 | 0.980 | 2
      | Test    | 0.645 | 0.696 | 2
---------------------------------------------
1     | Train   | 0.937 | 0.983 | 2
      | Test    | 0.477 | 0.525 | 2
---------------------------------------------
2     | Train   | 0.927 | 0.981 | 2
      | Test    | 0.531 | 0.551 | 2
---------------------------------------------
3     | Train   | 0.940 | 0.982 | 2
      | Test    | 0.539 | 0.541 | 2
---------------------------------------------
4     | Train   | 0.941 | 0.984 | 2
      | Test    | 0.516 | 0.520 | 2
---------------------------------------------
5     | Train   | 0.925 | 0.981 | 2
      | Test    | 0.566 | 0.584 | 2
---------------------------------------------


In [ ]:
from cuml.linear_model import LogisticRegression